### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

c:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
## step1 : Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(raw_docs)


In [5]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)\nAt the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standard patterns like Stuff, Map-Reduce, and Refine. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain integrates seamlessly with vector databases like FAISS, 

In [6]:
### step 2: Vector Store
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embedding_model)

## step 3:MMR Retriever
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7914.07it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E9950BF920>, search_type='mmr', search_kwargs={'k': 5})

In [45]:
## step 4 : LLM and Prompt

import os
from dotenv import load_dotenv
from langchain_ollama import OllamaEmbeddings


os.environ.pop("GROQ_API_KEY", None)
load_dotenv(override=True)
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm = init_chat_model(model="openai/gpt-oss-120b", model_provider="groq", api_key=os.getenv("GROQ_API_KEY"))
llm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E99C51E8D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E999E0F0E0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [46]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain=query_expansion_prompt| llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E99C51E8D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E999E0F0E0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)
| StrOutputParser()

In [47]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'**Expanded query**\n\n```\n(LangChain OR "Lang Chain" OR "LangChain framework")\nAND\n(memory OR "state management" OR "context retention" OR "conversation memory" OR "memory buffer" OR "memory store"\n     OR "persistent memory" OR "short‑term memory" OR "long‑term memory" OR caching)\nAND\n("ConversationBufferMemory" OR "ConversationSummaryMemory" OR "ConversationBufferWindowMemory"\n     OR "ConversationTokenBufferMemory" OR "VectorStoreRetrieverMemory" OR "RedisMemory"\n     OR "SQLMemory" OR "MongoDBMemory" OR "FAISS" OR "Chroma" OR "Pinecone" OR "Milvus")\nAND\n(LLM OR "large language model" OR "ChatGPT" OR "OpenAI" OR "LLMChain" OR agents)\nAND\n("retrieval‑augmented generation" OR RAG OR "prompt engineering")\n```'

In [48]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [49]:
# Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

In [51]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

**Expanded query**

```json
{
  "input": "What memory types and storage mechanisms does the LangChain framework support?  Specifically, which memory classes, back‑ends, and technical implementations are available for managing short‑term (conversation buffer, chat history, session state), long‑term (persistent, vector‑store, database‑backed) and hybrid memory in LangChain?  Include synonyms such as “state management,” “context storage,” “memory modules,” “memory providers,” and related terms like “retrieval‑augmented generation,” “vector store memory (FAISS, Chroma, Pinecone, Weaviate, Milvus),” “summary memory,” “conversation buffer memory,” “entity memory,” “SQL/NoSQL memory,” “in‑memory cache,” “persistent cache,” “agent memory,” and “LLM chain memory.”  Also look for documentation, examples, and technical details on how these memory components are integrated and configured within LangChain."
}
```
✅ Answer:
 LangChain provides built‑in memory modules that let an LLM keep track of wh

In [53]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

**Expanded query**

```
CrewAI agents 
OR CrewAI autonomous agents 
OR CrewAI AI bots 
OR CrewAI digital crew members 
OR CrewAI virtual crew agents 
OR multi‑agent AI system 
OR collaborative AI agents 
OR role‑based AI agents 
OR LLM‑powered agents 
OR GPT‑4 agents in CrewAI 
OR AI‑driven crew management 
OR AI crew coordination 
OR agent orchestration platform 
OR CrewAI framework 
OR AI agent architecture 
OR agent‑based modeling 
OR reinforcement‑learning agents in CrewAI 
OR knowledge‑graph‑enabled agents 
OR prompt‑engineering for CrewAI agents 
OR task delegation in CrewAI 
OR human‑in‑the‑loop agents in CrewAI 
OR AI crew simulation 
OR AI‑driven team automation 
OR CrewAI vs other multi‑agent frameworks 
OR AI‑powered crew workflow automation
```
✅ Answer:
 **CrewAI agents** are the building‑blocks of a CrewAI “crew.”  
Each agent is a self‑contained LLM‑powered entity that:

| Aspect | What it means in CrewAI |
|--------|------------------------|
| **Purpose / Goal** | Every